# Skin Cancer Detection — Phase 3 corrigée (anti-NaN)
**EfficientNetV2-S + CBAM multi-échelle** | ISIC2019 (+ HAM10000 dédupliqué) | validation externe PH2

Ce notebook **ne refait pas** les phases 1, 2a et 2b : il repart du checkpoint *step2* produit par le notebook `skincanerf`
et reprend uniquement la phase 3 (mixup + échantillonnage équilibré), avec :
- les couches CBAM calculées en **float32** (cause probable des NaN en mixed precision) ;
- une **reprise automatique** depuis le meilleur checkpoint si un NaN survient malgré tout ;
- les **mêmes splits** que le run d'origine (relus depuis les CSV, pas de re-split → pas de fuite) ;
- ensemble des checkpoints choisi sur VAL, calibration, PH2 calibré et intervalles de confiance.

**Inputs requis** : Output du notebook `skincanerf` + datasets ISIC2019 (cdeotte), HAM10000 (surajghuwalewala), PH2 (spacesurfer).
**Accelerator** : GPU T4 x2. **Durée estimée** : ~4 h 30.

## Step 1 — Setup : seeds, GPU, mixed precision, constantes

In [1]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

mixed_precision.set_global_policy("mixed_float16")

# ─── Constantes globales ───────────────────────────────────
IMG_SIZE   = 384
BATCH_SIZE = 32      
EVAL_RESIZE  = int(IMG_SIZE * 1.15)   # resize avant center-crop à l'inférence
NUM_CLASSES  = 7
AUTOTUNE     = tf.data.AUTOTUNE

# Régularisation
LABEL_SMOOTH = 0.1
WEIGHT_DECAY = 1e-4
MIXUP_ALPHA  = 0.1

# Mapping stable (utilisé partout dans le projet)
CLASSES      = ["akiec", "bcc", "bkl", "df", "nv", "mel", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_NAMES_FULL = {
    "akiec": "Actinic keratoses",
    "bcc":   "Basal cell carcinoma",
    "bkl":   "Benign keratosis-like lesions",
    "df":    "Dermatofibroma",
    "nv":    "Melanocytic nevi",
    "mel":   "Melanoma",
    "vasc":  "Vascular lesions",
}

print("TF:", tf.__version__)
print("GPUs:", gpus)
print("Mixed precision:", mixed_precision.global_policy())
print("Mapping:", CLASS_TO_IDX)


assert len(gpus) > 0, "❌ PAS DE GPU → Settings → Accelerator → GPU T4 x2"
if len(gpus) < 2:
    print("⚠️  1 seul GPU détecté : ça marche, mais ~2x plus lent (choisis GPU T4 x2).")

# Output du notebook d'entraînement d'origine (checkpoints + splits)
INPUT_DIR = "/kaggle/input/notebooks/rihembousbih/skincanerf"
assert os.path.exists(INPUT_DIR), f"❌ {INPUT_DIR} introuvable → + Add Input → Notebooks → skincanerf"
OUT_DIR   = "/kaggle/working"

strategy = tf.distribute.MirroredStrategy(
    cross_device_ops=tf.distribute.ReductionToOneDevice())
print("Répliques:", strategy.num_replicas_in_sync)   # doit afficher 2

TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision: <DTypePolicy "mixed_float16">
Mapping: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'nv': 4, 'mel': 5, 'vasc': 6}
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Répliques: 2


I0000 00:00:1790114407.499627      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790114407.502606      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


## Step 2 — Couches CBAM (calcul en float32 → anti-NaN)
`__init__` et `build` sont inchangés : les poids des anciens checkpoints se rechargent tels quels.

In [2]:
from tensorflow.keras import layers, Model

class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.add = layers.Add()
        self.act = layers.Activation("sigmoid")
        self.mul = layers.Multiply()

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden   = max(channels // self.ratio, 1)
        self.d1  = layers.Dense(hidden, activation="relu",
                                kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d2  = layers.Dense(channels,
                                kernel_initializer="he_normal", use_bias=True, dtype="float32")
        self.d1.build((None, 1, 1, channels))
        self.d2.build((None, 1, 1, hidden))
        super().build(input_shape)

    def call(self, x):                                   # calcul en float32
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        c   = int(x.shape[-1])
        avg = tf.reshape(tf.reduce_mean(x32, axis=[1, 2]), (-1, 1, 1, c))
        mx  = tf.reshape(tf.reduce_max(x32,  axis=[1, 2]), (-1, 1, 1, c))
        att = tf.sigmoid(self.d2(self.d1(avg)) + self.d2(self.d1(mx)))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        cfg = super().get_config(); cfg.update({"ratio": self.ratio}); return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.mul = layers.Multiply()

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding="same", activation="sigmoid",
                                  kernel_initializer="he_normal", use_bias=False, dtype="float32")
        self.conv.build((input_shape[0], input_shape[1], input_shape[2], 2))
        super().build(input_shape)

    def call(self, x):                                   # calcul en float32
        in_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        avg = tf.reduce_mean(x32, axis=-1, keepdims=True)
        mx  = tf.reduce_max(x32,  axis=-1, keepdims=True)
        att = self.conv(tf.concat([avg, mx], axis=-1))
        return tf.cast(x32 * att, in_dtype)

    def get_config(self):
        cfg = super().get_config(); cfg.update({"kernel_size": self.kernel_size}); return cfg


CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention, "SpatialAttention": SpatialAttention}
print("CBAM (float32) défini.")

CBAM (float32) défini.


## Step 3 — Pipeline tf.data (identique au run d'origine)
Loss utilisée : entropie croisée + label smoothing (pas de focal loss).

In [3]:
class NanWatch(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch % 50:
            return
        for v in self.model.variables:
            if not bool(tf.reduce_all(tf.math.is_finite(tf.cast(v, tf.float32)))):
                print(f"\n⚠️  NON-FINI dans « {v.path} » au batch {batch}")
                self.model.stop_training = True
                return

In [4]:
PREPROCESS = lambda x: x        # EfficientNetV2 : preprocessing intégré au modèle

def decode_image(path):
    """Décode jpg/png/bmp selon l'extension."""
    img_bytes = tf.io.read_file(path)
    ext = tf.strings.lower(tf.strings.split(path, ".")[-1])

    def _jpg(): return tf.image.decode_jpeg(img_bytes, channels=3)
    def _png(): return tf.image.decode_png(img_bytes,  channels=3)
    def _bmp(): return tf.image.decode_bmp(img_bytes)

    img = tf.case(
        [(tf.equal(ext, "jpg"),  _jpg),
         (tf.equal(ext, "jpeg"), _jpg),
         (tf.equal(ext, "png"),  _png),
         (tf.equal(ext, "bmp"),  _bmp)],
        default=_jpg, exclusive=True
    )
    return tf.ensure_shape(img, [None, None, 3])


# ── NOUVEAU : normalisation d'illuminant (Shades of Gray, p=6) ──
# Neutralise la balance des blancs du dermatoscope. C'est le correctif
# principal pour l'écart de généralisation observé sur PH2.
def shades_of_gray(img, p=6.0):
    """img : float32 en échelle 0-255. Retourne float32 0-255."""
    flat  = tf.reshape(img, [-1, 3])
    illum = tf.pow(tf.reduce_mean(tf.pow(flat + 1e-6, p), axis=0), 1.0 / p)
    illum = illum / (tf.norm(illum) + 1e-6)
    img   = img / (illum * tf.sqrt(3.0) + 1e-6)
    return tf.clip_by_value(img, 0.0, 255.0)


def augment_train(img):
    """img : float32, 0-255, AVANT resnet_preprocess."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))

    # MODIFIÉ : amplitudes doublées pour couvrir la variabilité inter-appareils
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.30)      # était 0.15
    img = tf.image.random_contrast(img,   0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_saturation(img, 0.6, 1.4)  # était 0.8, 1.2
    img = tf.image.random_hue(img,        0.08)      # était 0.03
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0

    # cutout à forme statique : masque booléen sur une grille fixe
    h  = tf.random.uniform([], IMG_SIZE//8, IMG_SIZE//4, dtype=tf.int32)
    y0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, IMG_SIZE - h, dtype=tf.int32)
    yy = tf.range(IMG_SIZE)[:, None]
    xx = tf.range(IMG_SIZE)[None, :]
    inside = (yy >= y0) & (yy < y0 + h) & (xx >= x0) & (xx < x0 + h)
    apply_cut = tf.cast(tf.random.uniform([]) < 0.5, tf.float32)
    keep = 1.0 - apply_cut * tf.cast(inside, tf.float32)[:, :, None]
    img = img * keep
    return tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])


def load_and_preprocess(path, label, training=False):
    img = tf.cast(decode_image(path), tf.float32)

    if training:
        shape = tf.shape(img)
        scale = tf.random.uniform([], 0.7, 1.0)
        h = tf.cast(tf.cast(shape[0], tf.float32) * scale, tf.int32)
        w = tf.cast(tf.cast(shape[1], tf.float32) * scale, tf.int32)
        img = tf.image.random_crop(img, [h, w, 3])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = augment_train(img)
    else:
        # MODIFIÉ : resize + center-crop, au lieu d'un resize direct.
        # L'entraînement voit des crops à 70-100 % ; un resize plein cadre
        # à l'inférence crée un décalage d'échelle systématique.
        img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
        off = (EVAL_RESIZE - IMG_SIZE) // 2
        img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)

    img   = shades_of_gray(img)          # NOUVEAU — train ET inférence
    img   = PREPROCESS(img)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label


def make_dataset(df, training=False, cache=False, shuffle_buffer=4096, batched=True):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values.astype(str), df["label"].values.astype(np.int32))
    )
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_and_preprocess(p, y, training=training),
                num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if batched:                              # NOUVEAU : option non-batchée
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


# ── NOUVEAU : mixup ──
def mixup(ds, alpha=MIXUP_ALPHA):
    """À appliquer APRÈS .batch(). Produit des labels mous."""
    def _mix(imgs, labels):
        b   = tf.shape(imgs)[0]
        g1  = tf.random.gamma([b], alpha)
        g2  = tf.random.gamma([b], alpha)
        lam = g1 / (g1 + g2)                  # Beta(alpha, alpha)
        idx = tf.random.shuffle(tf.range(b))
        li  = tf.reshape(lam, [b, 1, 1, 1])
        ll  = tf.reshape(lam, [b, 1])
        return (li * imgs   + (1 - li) * tf.gather(imgs,   idx),
                ll * labels + (1 - ll) * tf.gather(labels, idx))
    return ds.map(_mix, num_parallel_calls=AUTOTUNE)


# ── NOUVEAU : échantillonnage équilibré (remplace la duplication) ──
def balanced_dataset(df, power=0.5):
    """Poids par classe ∝ n^power. power=0.5 (racine) = compromis usuel ;
    power=0 = uniforme strict ; power=1 = distribution naturelle."""
    dss, weights = [], []
    for c in CLASSES:
        sub = df[df["dx"] == c]
        if len(sub) == 0:
            continue
        d = make_dataset(sub, training=True,
                         shuffle_buffer=min(len(sub), 4096), batched=False)
        dss.append(d.repeat())
        weights.append(float(len(sub)) ** power)
    w = np.array(weights) / np.sum(weights)
    ds = tf.data.Dataset.sample_from_datasets(dss, weights=list(w), seed=SEED)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = \
        tf.data.experimental.AutoShardPolicy.DATA
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE).with_options(opts)


print("Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).")

Pipeline tf.data défini (shades-of-gray + mixup + balanced sampling).


In [5]:
from sklearn.metrics import f1_score
from tensorflow.keras.optimizers.schedules import CosineDecay


class MacroF1(tf.keras.callbacks.Callback):
    """Calcule val_macro_f1 en fin d'époque. À placer EN PREMIER dans callbacks."""
    def __init__(self, val_ds, y_true):
        super().__init__()
        self.ds, self.y = val_ds, y_true

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = np.argmax(self.model.predict(self.ds, verbose=0), axis=1)
        logs["val_macro_f1"] = f1_score(self.y, p, average="macro")
        print(f"   val_macro_f1: {logs['val_macro_f1']:.4f}")


def make_callbacks(val_ds, y_val, ckpt_path, patience=10, best=None):
    """best : score à battre pour écraser le checkpoint (utile en cas de reprise)."""
    return [
        MacroF1(val_ds, y_val),
        NanWatch(),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_macro_f1",
                                           mode="max", save_best_only=True,
                                           initial_value_threshold=best),
        tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                         patience=patience,
                                         restore_best_weights=True),
    ]


def cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs=2):
    spe = max(n_train // BATCH_SIZE, 1)
    return CosineDecay(initial_learning_rate=peak_lr / 10,
                       decay_steps=spe * n_epochs,
                       warmup_target=peak_lr,
                       warmup_steps=spe * warmup_epochs,
                       alpha=0.01)


print("Callbacks et schedule définis.")
def make_optimizer(peak_lr, n_epochs, n_train, warmup_epochs=2):
    return tf.keras.optimizers.AdamW(
        learning_rate=cosine_lr(peak_lr, n_epochs, n_train, warmup_epochs),
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )

CE_SMOOTH = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

Callbacks et schedule définis.


## Step 4 — Recharger les splits d'origine (pas de re-split)
Le checkpoint step2 a été entraîné sur `split_train.csv` : il faut évaluer sur exactement les mêmes VAL/TEST, sinon fuite.

In [6]:
import pandas as pd

train_df = pd.read_csv(f"{INPUT_DIR}/split_train.csv")
val_df   = pd.read_csv(f"{INPUT_DIR}/split_val.csv")
test_df  = pd.read_csv(f"{INPUT_DIR}/split_test.csv")
ph2_df   = pd.read_csv(f"{INPUT_DIR}/split_ph2.csv")

# ─── Les images sont-elles accessibles ? ───
for nom, df in [("train", train_df), ("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
    manquants = (~df["path"].map(os.path.exists)).sum()
    print(f"{nom:5s}: {len(df):6d} images, {manquants} introuvables")
    assert manquants == 0, (f"❌ Images {nom} introuvables (ex: {df['path'].iloc[0]}) → "
                            "ajoute en Input les datasets ISIC2019 / HAM10000 / PH2")

# ─── Anti-fuite ───
for a, b, nom in [(train_df, val_df, "train/val"), (train_df, test_df, "train/test"),
                  (val_df, test_df, "val/test")]:
    n = len(set(a["lesion_id"]) & set(b["lesion_id"]))
    assert n == 0, f"❌ {n} lésions communes {nom}"
print("✅ 0 lésion commune entre train / val / test")

val_ds = make_dataset(val_df, training=False)
y_val, y_test, y_ph2 = (val_df["label"].values, test_df["label"].values, ph2_df["label"].values)
print("\nDistribution train :", train_df["dx"].value_counts().to_dict())

train:  17455 images, 0 introuvables
val  :   3707 images, 0 introuvables
test :   3738 images, 0 introuvables
ph2  :    197 images, 0 introuvables
✅ 0 lésion commune entre train / val / test

Distribution train : {'nv': 9005, 'mel': 3151, 'bcc': 2369, 'bkl': 1833, 'akiec': 753, 'df': 175, 'vasc': 169}


## Step 5 — Phase 3 : fine-tuning complet (mixup + échantillonnage équilibré), avec reprise auto sur NaN
~6 min / époque avec 2 T4 → ~4 h pour 40 époques (moins si early stopping).

In [7]:
import shutil
from tensorflow.keras.layers import BatchNormalization

STEP1_PATH   = f"{INPUT_DIR}/resnet50_cbam_finetune_step1.keras"   # ancien nom, poids EfficientNetV2-S
STEP2_PATH   = f"{INPUT_DIR}/resnet50_cbam_finetune_step2.keras"
FINAL_CKPT   = f"{OUT_DIR}/effnetv2s_cbam_final.keras"
EPOCHS_P3    = 40
PEAK_LR      = 2e-5
MAX_RESTARTS = 3

with strategy.scope():
    model = tf.keras.models.load_model(STEP2_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)

# Contrôle : le step2 rechargé doit redonner ~0.71-0.72 de macro-F1 VAL
p = model.predict(val_ds, verbose=0)
assert np.isfinite(p).all(), "NaN dès le chargement — ne continue pas."
f1_step2 = f1_score(y_val, p.argmax(1), average="macro")
print(f"step2 rechargé : val_macro_f1 = {f1_step2:.4f}")

# Point de départ : le checkpoint final = step2 ; il n'est écrasé que si la phase 3 fait mieux
shutil.copy(STEP2_PATH, FINAL_CKPT)

train_ds_bal = mixup(balanced_dataset(train_df, power=0.5))
STEPS_P3 = len(train_df) // BATCH_SIZE

with strategy.scope():
    for layer in model.layers:
        layer.trainable = not isinstance(layer, BatchNormalization)

best_f1, epochs_done, lr, historique = f1_step2, 0, PEAK_LR, []
for attempt in range(MAX_RESTARTS + 1):
    remaining = EPOCHS_P3 - epochs_done
    if remaining <= 0:
        break
    print(f"\n▶ Tentative {attempt + 1} : {remaining} époques, lr max = {lr:.1e}, "
          f"F1 à battre = {best_f1:.4f}")
    with strategy.scope():
        model.compile(optimizer=make_optimizer(lr, remaining, len(train_df),
                                               warmup_epochs=2 if attempt == 0 else 1),
                      loss=CE_SMOOTH, metrics=["accuracy"])
    h = model.fit(train_ds_bal, steps_per_epoch=STEPS_P3,
                  validation_data=val_ds, epochs=remaining,
                  callbacks=make_callbacks(val_ds, y_val, FINAL_CKPT, patience=10, best=best_f1),
                  verbose=1)
    historique.append(h.history)
    epochs_done += len(h.history["loss"])
    best_f1 = max([best_f1] + [v for v in h.history.get("val_macro_f1", []) if np.isfinite(v)])

    poids_ok = all(np.isfinite(w).all() for w in model.get_weights())
    if np.isfinite(h.history["loss"][-1]) and poids_ok:
        print(f"\n✅ Phase 3 terminée normalement ({epochs_done} époques).")
        break
    print(f"\n⚠️  NaN après {epochs_done} époques → rechargement du meilleur checkpoint, lr / 2")
    model.load_weights(FINAL_CKPT)
    lr /= 2
else:
    print("\n⚠️  Nombre max de reprises atteint — on garde le meilleur checkpoint.")

print(f"Meilleur val_macro_f1 phase 3 : {best_f1:.4f}  (step2 : {f1_step2:.4f})")

step2 rechargé : val_macro_f1 = 0.7206

▶ Tentative 1 : 40 époques, lr max = 2.0e-05, F1 à battre = 0.7206
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/ta

In [8]:
# ─── Contrôle du checkpoint final (sur disque) ───
model_final = tf.keras.models.load_model(FINAL_CKPT, custom_objects=CUSTOM_OBJECTS, compile=False)
p = model_final.predict(val_ds, verbose=0)
assert np.isfinite(p).all(), "❌ NaN dans le checkpoint final"
print(f"✅ Checkpoint final sain. val_macro_f1 (sans TTA) = {f1_score(y_val, p.argmax(1), average='macro'):.4f}")
del model_final

I0000 00:00:1790130858.917869      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


✅ Checkpoint final sain. val_macro_f1 (sans TTA) = 0.8052


## Step 6 — Fonctions d'évaluation (TTA, calibration, bootstrap)

In [9]:
import time
from scipy.optimize import minimize, differential_evolution
from sklearn.metrics import (accuracy_score, f1_score, recall_score, log_loss, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import label_binarize

MEL = CLASS_TO_IDX["mel"]
CIBLE_SENS_MEL = 0.85


def predict_tta(model, df, n_rot=4):
    """Moyenne sur 4 rotations x 2 flips, prétraitement identique à l'inférence."""
    probs = np.zeros((len(df), NUM_CLASSES), dtype=np.float64)
    for k in range(n_rot):
        for flip in (False, True):
            def _map(p, y, k=k, flip=flip):
                img = tf.cast(decode_image(p), tf.float32)
                img = tf.image.resize(img, (EVAL_RESIZE, EVAL_RESIZE))
                off = (EVAL_RESIZE - IMG_SIZE) // 2
                img = tf.image.crop_to_bounding_box(img, off, off, IMG_SIZE, IMG_SIZE)
                img = tf.image.rot90(img, k=k)
                if flip:
                    img = tf.image.flip_left_right(img)
                img = shades_of_gray(img)
                return PREPROCESS(img), y
            ds = (tf.data.Dataset.from_tensor_slices(
                      (df["path"].values.astype(str), df["label"].values.astype(np.int32)))
                  .map(_map, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH_SIZE).prefetch(AUTOTUNE))
            probs += model.predict(ds, verbose=0)
    return probs / (n_rot * 2)


def preds_cached(model, name, df, split):
    f = f"{OUT_DIR}/preds_{split}_{name}.npy"
    if os.path.exists(f):
        return np.load(f)
    t0 = time.time()
    p = predict_tta(model, df)
    np.save(f, p)
    print(f"  {split:4s}/{name:5s} : {time.time() - t0:.0f}s")
    return p


def ensemble_weighted(preds_list, weights):
    w = np.abs(np.array(weights, dtype=float))
    w = w / (w.sum() + 1e-12)
    return sum(w[i] * preds_list[i] for i in range(len(preds_list)))


def apply_temperature(probs, T):
    logits = np.log(np.clip(probs, 1e-12, 1 - 1e-12))
    s = np.exp(logits / T)
    return s / s.sum(axis=1, keepdims=True)


def find_temperature(probs_val, y):
    def nll(x):
        return log_loss(y, apply_temperature(probs_val, float(x[0])),
                        labels=np.arange(probs_val.shape[1]))
    return float(minimize(nll, x0=[1.0], bounds=[(0.05, 5.0)], method="L-BFGS-B").x[0])


def apply_thresholds(probs, thr):
    thr = np.clip(np.asarray(thr, dtype=float), 0.05, 5.0)
    return np.argmax(probs / thr[None, :], axis=1)


def optimize_thresholds(probs_val, y):
    """Maximise l'accuracy VAL sous contrainte sensibilité mélanome >= CIBLE_SENS_MEL."""
    def obj(thr):
        pred = apply_thresholds(probs_val, thr)
        sens = recall_score(y, pred, labels=[MEL], average="macro", zero_division=0)
        if sens < CIBLE_SENS_MEL:
            return 10.0 + (CIBLE_SENS_MEL - sens) * 100.0
        return -accuracy_score(y, pred)
    res = differential_evolution(obj, [(0.2, 3.0)] * probs_val.shape[1],
                                 maxiter=80, popsize=10, polish=False, seed=SEED)
    return np.clip(res.x, 0.2, 3.0)


# ─── Métriques + bootstrap ───
acc    = lambda yt, yp: accuracy_score(yt, yp)
mf1    = lambda yt, yp: f1_score(yt, yp, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0)
s_mel  = lambda yt, yp: ((yt == MEL) & (yp == MEL)).sum() / max((yt == MEL).sum(), 1)
sp_mel = lambda yt, yp: ((yt != MEL) & (yp != MEL)).sum() / max((yt != MEL).sum(), 1)
M7 = {"Accuracy": acc, "Macro-F1": mf1, "Sensibilité mel": s_mel, "Spécificité mel": sp_mel}


def bootstrap_ci(yt, yp, metric, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [metric(yt[i], yp[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


def report(nom, yt, yp, metrics=M7):
    print(f"\n=== {nom} ({len(yt)} images) ===")
    out = {}
    for mname, m in metrics.items():
        v = m(yt, yp); lo, hi = bootstrap_ci(yt, yp, m)
        out[mname] = (float(v), float(lo), float(hi))
        print(f"  {mname:18s} {v:.4f}   IC95% [{lo:.4f} – {hi:.4f}]")
    return out

print("Fonctions d'évaluation définies.")

Fonctions d'évaluation définies.


## Step 7 — Prédictions TTA des 3 checkpoints (VAL / TEST / PH2)
~25-30 min. Sauvegardées au fur et à mesure dans /kaggle/working.

In [10]:
MODEL_PATHS = {"step1": STEP1_PATH, "step2": STEP2_PATH, "final": FINAL_CKPT}
NAMES = list(MODEL_PATHS)
P = {"val": {}, "test": {}, "ph2": {}}

for name, path in MODEL_PATHS.items():
    m = tf.keras.models.load_model(path, custom_objects=CUSTOM_OBJECTS, compile=False)
    for split, df in [("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
        P[split][name] = preds_cached(m, name, df, split)
    del m

print("\n=== Modèles individuels (TTA, argmax brut) ===")
for name in NAMES:
    pv, pt = P["val"][name].argmax(1), P["test"][name].argmax(1)
    print(f"{name:6s}  VAL Acc={acc(y_val, pv):.4f} F1={mf1(y_val, pv):.4f}   "
          f"TEST Acc={acc(y_test, pt):.4f} F1={mf1(y_test, pt):.4f}")

  val /step1 : 289s
  test/step1 : 271s
  ph2 /step1 : 44s
  val /step2 : 302s
  test/step2 : 256s
  ph2 /step2 : 36s
  val /final : 290s
  test/final : 253s
  ph2 /final : 36s

=== Modèles individuels (TTA, argmax brut) ===
step1   VAL Acc=0.7963 F1=0.6511   TEST Acc=0.7935 F1=0.6385
step2   VAL Acc=0.8282 F1=0.7291   TEST Acc=0.8229 F1=0.7121
final   VAL Acc=0.8632 F1=0.8010   TEST Acc=0.8604 F1=0.7895


## Step 8 — Ensemble : poids optimisés sur VAL, choix final/ensemble décidé sur VAL

In [11]:
def neg_f1(w):
    return -mf1(y_val, ensemble_weighted([P["val"][n] for n in NAMES], w).argmax(1))

res = differential_evolution(neg_f1, bounds=[(0, 1)] * len(NAMES),
                             seed=SEED, maxiter=40, tol=1e-4, workers=1)
W_ens = np.abs(res.x) / np.abs(res.x).sum()

f1_val_final = mf1(y_val, P["val"]["final"].argmax(1))
f1_val_ens   = -res.fun
USE_ENSEMBLE = f1_val_ens > f1_val_final + 0.005        # marge contre le bruit
W = W_ens if USE_ENSEMBLE else np.array([0.0, 0.0, 1.0])

print("Poids ensemble (VAL) :", dict(zip(NAMES, W_ens.round(3))))
print(f"VAL macro-F1 — final seul : {f1_val_final:.4f} | ensemble : {f1_val_ens:.4f}")
print("→ Retenu :", "ENSEMBLE" if USE_ENSEMBLE else "MODÈLE FINAL SEUL", "(décidé sur VAL)")

probs_val  = ensemble_weighted([P["val"][n]  for n in NAMES], W)
probs_test = ensemble_weighted([P["test"][n] for n in NAMES], W)
probs_ph2  = ensemble_weighted([P["ph2"][n]  for n in NAMES], W)

Poids ensemble (VAL) : {'step1': np.float64(0.248), 'step2': np.float64(0.002), 'final': np.float64(0.75)}
VAL macro-F1 — final seul : 0.8010 | ensemble : 0.8016
→ Retenu : MODÈLE FINAL SEUL (décidé sur VAL)


## Step 9 — Temperature scaling + seuils (sensibilité mélanome ≥ 0.85), sur VAL

In [12]:
T   = find_temperature(probs_val, y_val)
thr = optimize_thresholds(apply_temperature(probs_val, T), y_val)
print(f"T = {T:.4f}")
print("Seuils :", "  ".join(f"{IDX_TO_CLASS[i]}={thr[i]:.2f}" for i in range(NUM_CLASSES)))

yv_raw = probs_val.argmax(1)
yv_cal = apply_thresholds(apply_temperature(probs_val, T), thr)
for nom, yp in [("brut", yv_raw), ("calibré", yv_cal)]:
    print(f"VAL {nom:8s} Acc={acc(y_val, yp):.4f}  F1={mf1(y_val, yp):.4f}  "
          f"SensMel={s_mel(y_val, yp):.4f}  SpecMel={sp_mel(y_val, yp):.4f}")

T = 0.7853
Seuils : akiec=1.51  bcc=1.94  bkl=2.11  df=1.87  nv=2.57  mel=0.39  vasc=2.23
VAL brut     Acc=0.8632  F1=0.8010  SensMel=0.7254  SpecMel=0.9613
VAL calibré  Acc=0.8384  F1=0.7909  SensMel=0.8502  SpecMel=0.8906


## Step 10 — TEST (une seule fois) : brut vs calibré, avec IC 95 %

In [13]:
target_names = [CLASS_NAMES_FULL[IDX_TO_CLASS[i]] for i in range(NUM_CLASSES)]
yt_raw = probs_test.argmax(1)
yt_cal = apply_thresholds(apply_temperature(probs_test, T), thr)

res_test_raw = report("TEST brut",    y_test, yt_raw)
res_test_cal = report("TEST calibré", y_test, yt_cal)

yb = label_binarize(y_test, classes=np.arange(NUM_CLASSES))
print(f"\nMacro AUC (one-vs-rest) : {roc_auc_score(yb, probs_test, average='macro'):.4f}")

for nom, yp in [("brut", yt_raw), ("calibré", yt_cal)]:
    print(f"\n--- Rapport par classe ({nom}) ---")
    print(classification_report(y_test, yp, labels=np.arange(NUM_CLASSES),
                                target_names=target_names, digits=4, zero_division=0))
print("Matrice de confusion (calibré ; lignes = vrai, colonnes = prédit) :")
print(confusion_matrix(y_test, yt_cal, labels=np.arange(NUM_CLASSES)))


=== TEST brut (3738 images) ===
  Accuracy           0.8604   IC95% [0.8502 – 0.8708]
  Macro-F1           0.7895   IC95% [0.7585 – 0.8155]
  Sensibilité mel    0.7391   IC95% [0.7059 – 0.7716]
  Spécificité mel    0.9613   IC95% [0.9545 – 0.9679]

=== TEST calibré (3738 images) ===
  Accuracy           0.8285   IC95% [0.8170 – 0.8406]
  Macro-F1           0.7698   IC95% [0.7359 – 0.7971]
  Sensibilité mel    0.8507   IC95% [0.8240 – 0.8765]
  Spécificité mel    0.8835   IC95% [0.8725 – 0.8948]

Macro AUC (one-vs-rest) : 0.9741

--- Rapport par classe (brut) ---
                               precision    recall  f1-score   support

            Actinic keratoses     0.7047    0.6863    0.6954       153
         Basal cell carcinoma     0.8773    0.9167    0.8966       468
Benign keratosis-like lesions     0.7421    0.7287    0.7353       387
               Dermatofibroma     0.7600    0.6786    0.7170        28
             Melanocytic nevi     0.9070    0.9327    0.9197      1976
   

## Step 11 — PH2 (validation externe) : brut vs calibré, avec IC 95 %
Tâche binaire mélanome vs non-mélanome (PH2 ne contient que nv / mel).

In [14]:
yt_bin = (y_ph2 == MEL).astype(int)
M_bin = {"Accuracy": acc,
         "Sensibilité mel": lambda a, b: ((a == 1) & (b == 1)).sum() / max((a == 1).sum(), 1),
         "Spécificité mel": lambda a, b: ((a == 0) & (b == 0)).sum() / max((a == 0).sum(), 1)}

yp_raw = (probs_ph2.argmax(1) == MEL).astype(int)
yp_cal = (apply_thresholds(apply_temperature(probs_ph2, T), thr) == MEL).astype(int)
res_ph2_raw = report("PH2 brut",    yt_bin, yp_raw, M_bin)
res_ph2_cal = report("PH2 calibré", yt_bin, yp_cal, M_bin)
auc_ph2 = roc_auc_score(yt_bin, probs_ph2[:, MEL])
print(f"\nAUC mélanome PH2 : {auc_ph2:.4f}")
print("Matrice (calibré ; lignes = vrai [non-mel, mel]) :")
print(confusion_matrix(yt_bin, yp_cal))


=== PH2 brut (197 images) ===
  Accuracy           0.8528   IC95% [0.8020 – 0.9036]
  Sensibilité mel    0.5577   IC95% [0.4200 – 0.6923]
  Spécificité mel    0.9586   IC95% [0.9221 – 0.9868]

=== PH2 calibré (197 images) ===
  Accuracy           0.8477   IC95% [0.7970 – 0.8985]
  Sensibilité mel    0.6731   IC95% [0.5416 – 0.8036]
  Spécificité mel    0.9103   IC95% [0.8627 – 0.9542]

AUC mélanome PH2 : 0.8590
Matrice (calibré ; lignes = vrai [non-mel, mel]) :
[[132  13]
 [ 17  35]]


## Step 12 — Sauvegarde des artefacts

In [15]:
import json

np.save(f"{OUT_DIR}/ensemble_weights.npy", W)
np.save(f"{OUT_DIR}/temperature.npy", np.array([T]))
np.save(f"{OUT_DIR}/thresholds.npy", thr)
np.save(f"{OUT_DIR}/probs_val.npy",  probs_val)
np.save(f"{OUT_DIR}/probs_test.npy", probs_test)
np.save(f"{OUT_DIR}/probs_ph2.npy",  probs_ph2)

for nom, df in [("train", train_df), ("val", val_df), ("test", test_df), ("ph2", ph2_df)]:
    df.to_csv(f"{OUT_DIR}/split_{nom}.csv", index=False)

pd.DataFrame({"image_uid": test_df["image_uid"].values,
              "true": [IDX_TO_CLASS[i] for i in y_test],
              "pred_raw": [IDX_TO_CLASS[i] for i in yt_raw],
              "pred_cal": [IDX_TO_CLASS[i] for i in yt_cal],
              "confidence": apply_temperature(probs_test, T).max(1)}
            ).to_csv(f"{OUT_DIR}/predictions_test.csv", index=False)

config = {
    "backbone": "EfficientNetV2-S + CBAM multi-échelle (c4 + c5)",
    "loss": "categorical crossentropy + label smoothing 0.1 (pas de focal loss)",
    "phase3": {"epochs_prevues": EPOCHS_P3, "epochs_effectuees": int(epochs_done),
               "best_val_macro_f1": float(best_f1), "val_macro_f1_step2": float(f1_step2)},
    "modeles": NAMES, "poids": W.tolist(), "ensemble_retenu": bool(USE_ENSEMBLE),
    "temperature": float(T), "seuils": thr.tolist(), "cible_sens_mel": CIBLE_SENS_MEL,
    "test_brut": res_test_raw, "test_calibre": res_test_cal,
    "ph2_brut": res_ph2_raw, "ph2_calibre": res_ph2_cal, "ph2_auc_mel": float(auc_ph2),
}
with open(f"{OUT_DIR}/run_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:40s} {os.path.getsize(f'{OUT_DIR}/{f}') / 1e6:8.2f} MB")

  __notebook__.ipynb                           9.62 MB
  effnetv2s_cbam_final.keras                 417.15 MB
  ensemble_weights.npy                         0.00 MB
  predictions_test.csv                         0.16 MB
  preds_ph2_final.npy                          0.01 MB
  preds_ph2_step1.npy                          0.01 MB
  preds_ph2_step2.npy                          0.01 MB
  preds_test_final.npy                         0.21 MB
  preds_test_step1.npy                         0.21 MB
  preds_test_step2.npy                         0.21 MB
  preds_val_final.npy                          0.21 MB
  preds_val_step1.npy                          0.21 MB
  preds_val_step2.npy                          0.21 MB
  probs_ph2.npy                                0.01 MB
  probs_test.npy                               0.21 MB
  probs_val.npy                                0.21 MB
  run_config.json                              0.00 MB
  split_ph2.csv                                0.03 MB
  split_te

## Step 13 — Compromis sensibilité / spécificité (seuils choisis sur VAL, appliqués au TEST)

In [16]:
probs_val_T = apply_temperature(probs_val, T)
print("Cible sens. mel | Acc test | Macro-F1 | Sens. mel test | Spéc. mel test")
for cible in [0.70, 0.75, 0.80, 0.85, 0.90]:
    CIBLE_SENS_MEL = cible
    t  = optimize_thresholds(probs_val_T, y_val)
    yp = apply_thresholds(apply_temperature(probs_test, T), t)
    print(f"     {cible:.2f}      |  {acc(y_test, yp):.4f}  |  {mf1(y_test, yp):.4f}  |"
          f"     {s_mel(y_test, yp):.4f}     |    {sp_mel(y_test, yp):.4f}")
CIBLE_SENS_MEL = 0.85

Cible sens. mel | Acc test | Macro-F1 | Sens. mel test | Spéc. mel test
     0.70      |  0.8590  |  0.7859  |     0.7507     |    0.9570
     0.75      |  0.8609  |  0.7962  |     0.7739     |    0.9511
     0.80      |  0.8456  |  0.7619  |     0.8043     |    0.9245
     0.85      |  0.8285  |  0.7698  |     0.8507     |    0.8835
     0.90      |  0.7945  |  0.7553  |     0.8884     |    0.8222
